In [1]:
import torch
from transformers import AutoTokenizer,AutoModel, AdamW
from torch.utils.data import DataLoader, Dataset, random_split
import pandas as pd
from tqdm import tqdm
import random
from transformers import BertForSequenceClassification

c:\Users\wangy\.conda\envs\machine\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from modelscope import snapshot_download
model_dir = snapshot_download('tiansz/bert-base-chinese')

2024-06-20 12:36:39,874 - modelscope - INFO - PyTorch version 2.2.2 Found.
2024-06-20 12:36:39,877 - modelscope - INFO - Loading ast index from C:\Users\wangy\.cache\modelscope\ast_indexer
2024-06-20 12:36:40,064 - modelscope - INFO - Loading done! Current index file version is 1.15.0, with md5 934c7804ee76f2bf1b2b2b583a1fc23e and a total number of 980 components indexed


In [3]:
# 加载预训练的BERT模型和分词器
tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = BertForSequenceClassification.from_pretrained(model_dir,num_labels=3)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at C:\Users\wangy\.cache\modelscope\hub\tiansz\bert-base-chinese and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [4]:
class SentimentDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=128):
        self.dataframe = dataframe
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        text = self.dataframe.iloc[idx]['review']
        label = self.dataframe.iloc[idx]['label']
        encoding = self.tokenizer(text, padding='max_length', truncation=True, max_length=self.max_length, return_tensors='pt')
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),  
            'labels': torch.tensor(label, dtype=torch.long)
    }

In [5]:
with open("train data\data.txt", "r", encoding="utf-8") as file:
    lines = file.readlines()

data = []
for line in lines:
    parts = line.strip().split("\t")
    if len(parts) == 2:
        label, text = parts
        data.append((int(label), text))
    else:
        # 可以选择记录错误、跳过该行或采取其他措施
        print(f"Skipping line due to unexpected format: {line}")

df = pd.DataFrame(data, columns=["label", "review"])

dataset = SentimentDataset(df, tokenizer, max_length=128)

# 划分训练集和验证集
train_size = int(0.85 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

# 创建数据加载器
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)

Skipping line due to unexpected format: 	牛市呢？？牛市呢？只剩下牛

Skipping line due to unexpected format: 	越跌越买，我一万个理由都不相信他下星期一还会跌，拭目以待，我今天百分之百会满仓干



In [6]:
# 设置训练参数
optimizer = AdamW(model.parameters(), lr=1e-5)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

c:\Users\wangy\.conda\envs\machine\Lib\site-packages\transformers\optimization.py:521: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(21128, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [7]:
# 训练模型
model.train()
for epoch in range(3):
    for batch in tqdm(train_loader, desc="Epoch {}".format(epoch + 1)):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

Epoch 3: 100%|██████████| 1700/1700 [03:18<00:00,  8.56it/s]


In [8]:
torch.save(model.state_dict(), 'model.pth')

In [9]:
# 验证集下评估模型
model.eval()
total_eval_accuracy = 0
for batch in tqdm(val_loader, desc="Evaluating"):
    input_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    labels = batch['labels'].to(device)

    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        logits = outputs.logits
        preds = torch.argmax(logits, dim=1)
        accuracy = (preds == labels).float().mean()
        total_eval_accuracy += accuracy.item()

average_eval_accuracy = total_eval_accuracy / len(val_loader)
print("Validation Accuracy:", average_eval_accuracy)


Evaluating: 100%|██████████| 300/300 [00:09<00:00, 30.08it/s]

Validation Accuracy: 0.7358333333333333


In [10]:
# 使用微调后的模型进行预测
def predict_sentiment(sentence):
    inputs = tokenizer(sentence, padding='max_length', truncation=True, max_length=128, return_tensors='pt').to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    logits = outputs.logits
    probs = torch.softmax(logits, dim=1)
    return probs[0]  # 返回所有类别的概率

def predict(sentence):
    probs = predict_sentiment(sentence)
    label = torch.argmax(probs).item()  # 获取最高概率的索引作为预测标签
    if label == 2:
        print("正面")
    elif label == 1:
        print("中性")
    else:
        print("负面")

predict("中原海控太强了")

正面


In [11]:
import torch
from torch.utils.data import Dataset

# 定义用于加载测试数据集的类
class TestDataset(Dataset):
    def __init__(self, file_path, tokenizer, max_length=128):
        self.data = []
        with open(file_path, "r", encoding="utf-8") as file:
            for line in file:
                label, text = line.strip().split("\t")
                self.data.append((int(label), text))
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        label, text = self.data[idx]
        encoding = self.tokenizer(text, padding='max_length', truncation=True, max_length=self.max_length, return_tensors='pt')
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# 加载测试数据集
test_dataset = TestDataset("train data\\test.txt", tokenizer, max_length=128)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=8, shuffle=False)

# 评估模型
model.eval()
total_eval_accuracy = 0
for batch in tqdm(test_loader, desc="Evaluating"):
    input_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    labels = batch['labels'].to(device)

    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
    
    logits = outputs.logits
    preds = torch.argmax(logits, dim=1)
    accuracy = (preds == labels).float().mean()
    total_eval_accuracy += accuracy.item()

average_eval_accuracy = total_eval_accuracy / len(test_loader)
print("Test Accuracy:", average_eval_accuracy)


Evaluating: 100%|██████████| 202/202 [00:06<00:00, 30.77it/s]

Test Accuracy: 0.7214462521052597
